# Texas Interconnection Timelines

Analysis of solar interconnection timelines in Texas, prepared for Nick & Matt's May 2026 meeting with Texas lobbyist on the need for IX reform.

Compares TX state-level IX timelines against other states, and breaks down performance by utility within TX.

In [ ]:
import pandas as pd

state_timelines   = pd.read_csv("../output_csvs/solartrace_timelines_by_state.csv")
utility_timelines = pd.read_csv("../output_csvs/solartrace_timelines_by_utility.csv")

STATE = "TX"

## Interconnection Timelines — 2023
### State-level: pre-install IX, final IX to PTO, total IX, and project time

In [ ]:
YEAR = 2023
MIN_INSTALLS = 100
METRICS = ["pre_install_ix_days", "final_ix_to_pto_days", "total_ix_days", "project_time_days"]
METRIC_LABELS = {
    "pre_install_ix_days":  "Pre-Install IX (days)",
    "final_ix_to_pto_days": "Final IX to PTO (days)",
    "total_ix_days":        "Total IX (days)",
    "project_time_days":    "Project Time (days)",
}

base = state_timelines[state_timelines["year"] == YEAR].copy()
base["total_ix_days"] = base["pre_install_ix_days"].where(
    base["pre_install_ix_days"].notna() & base["final_ix_to_pto_days"].notna()
) + base["final_ix_to_pto_days"].where(
    base["pre_install_ix_days"].notna() & base["final_ix_to_pto_days"].notna()
)

rows = []
for (size, tech), group in base.groupby(["size_class", "tech_class"]):
    tx_row = group[group["state"] == STATE]
    others = group[(group["state"] != STATE) & (group["installs"] >= MIN_INSTALLS)]

    row = {
        "Size": {"0_10kw": "0-10 kW", "10_20kw": "10-20 kW"}[size],
        "Tech": {"pv_only": "Solar Only", "pv_storage": "Solar+Storage"}[tech],
        f"{STATE} Installs": tx_row["installs"].values[0] if not tx_row.empty else float("nan"),
        "n other states": len(others),
    }
    for m in METRICS:
        lbl = METRIC_LABELS[m]
        row[f"{STATE} — {lbl}"] = tx_row[m].values[0] if not tx_row.empty else float("nan")
        row[f"Other States — {lbl}"] = others[m].median()
    rows.append(row)

comparison = pd.DataFrame(rows).sort_values(["Size", "Tech"]).reset_index(drop=True)

print(f"{STATE} vs. median of other states (≥{MIN_INSTALLS} installs, unweighted) — {YEAR}")
display(comparison)

## Utility-level analysis — TX 2023, 0-10kW, Solar Only

**Note on installs:** The utility-level data comes from the AHJ-Utility Timelines sheet, where each row is an AHJ-utility *pair*. AHJs served by multiple utilities appear more than once, so installs cannot be summed to a state total (TX has 284 such AHJs). The `Installs` column is useful for comparing relative utility size and as a weight for medians, but not for aggregation.

In [ ]:
UTIL_MIN_INSTALLS = 100

tx_util = utility_timelines[
    (utility_timelines["state"] == STATE) &
    (utility_timelines["year"] == YEAR) &
    (utility_timelines["size_class"] == "0_10kw") &
    (utility_timelines["tech_class"] == "pv_only") &
    (utility_timelines["installs"] >= UTIL_MIN_INSTALLS)
].copy()

tx_util["total_ix_days"] = tx_util["pre_install_ix_days"].where(
    tx_util["pre_install_ix_days"].notna() & tx_util["final_ix_to_pto_days"].notna()
) + tx_util["final_ix_to_pto_days"].where(
    tx_util["pre_install_ix_days"].notna() & tx_util["final_ix_to_pto_days"].notna()
)

# --- 1. Ranked utility list ---
ranked = tx_util[[
    "utility", "installs", "pre_install_ix_days", "final_ix_to_pto_days", "total_ix_days"
]].rename(columns={
    "utility":              "Utility",
    "installs":             "Installs",
    "pre_install_ix_days":  "Pre-Install IX (days)",
    "final_ix_to_pto_days": "Final IX to PTO (days)",
    "total_ix_days":        "Total IX (days)",
}).sort_values("Installs", ascending=False).reset_index(drop=True)

print(f"TX utilities by installs — {YEAR}, 0-10kW, Solar Only (≥{UTIL_MIN_INSTALLS} installs)")
display(ranked)

# --- 2. Spread summary ---
spread_metrics = {
    "Pre-Install IX (days)":  "pre_install_ix_days",
    "Final IX to PTO (days)": "final_ix_to_pto_days",
    "Total IX (days)":        "total_ix_days",
}

spread_rows = []
for label, col in spread_metrics.items():
    s = tx_util[col].dropna()
    spread_rows.append({
        "Metric":      label,
        "n utilities": len(s),
        "Min":         s.min(),
        "P25":         s.quantile(0.25),
        "Median":      s.median(),
        "P75":         s.quantile(0.75),
        "Max":         s.max(),
    })

spread = pd.DataFrame(spread_rows)
print(f"\nSpread across TX utilities — {YEAR}, 0-10kW, Solar Only (≥{UTIL_MIN_INSTALLS} installs)")
display(spread)